### Dispatcher de PDFs — EPE-SEGOV/MS, Resoluções Agesan-RS

Notebook único, parametrizado por `fonte`. Cada fonte é uma página estática
com lista de links `.pdf`, sem paginação. Deduplicação via manifesto.


In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml pypdf
dbutils.library.restartPython()


In [0]:
# Carrega a função compartilhada que atualiza o status real da fonte na
# tabela de controle a cada execução.
%run "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/scripts/utils_controle"


In [0]:
import os
import re
import io
import json
import time
import random
import hashlib
import unicodedata
from datetime import datetime, timezone
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests
from pypdf import PdfReader


In [0]:
try:
    dbutils.widgets.remove("fonte")
except Exception:
    pass

dbutils.widgets.text("fonte", "todas")
NOME_FONTE = dbutils.widgets.get("fonte")

CONFIGS_FONTES = {
    "epe_segov_ms": {
        "site_url": "https://www.epe.segov.ms.gov.br/publicacoes/",
        "source_id": "epe_segov_ms",
        "tema": "GERAL",
        "source_descricao": "Linked from EPE-SEGOV/MS — Publicações",
    },
    "agesan_resolucoes": {
        "site_url": "https://agesan-rs.com.br/documentacao/resolucoes/",
        "source_id": "agesan_rs_resolucoes",
        "tema": "SANEAMENTO",
        "source_descricao": "Linked from Agesan-RS — Resoluções",
    },
    "agesan_resolucoes_csr": {
        "site_url": "https://agesan-rs.com.br/documentacao/resolucoes-csr/",
        "source_id": "agesan_rs_resolucoes_csr",
        "tema": "SANEAMENTO",
        "source_descricao": "Linked from Agesan-RS — Resoluções CSR",
    },
    "agesan_resolucoes_dc": {
        "site_url": "https://agesan-rs.com.br/documentacao/resolucoes-csr-2/",
        "source_id": "agesan_rs_resolucoes_dc",
        "tema": "SANEAMENTO",
        "source_descricao": "Linked from Agesan-RS — Resoluções DC",
    },
    "ccee_atas_diretoria": {
        # Liferay (CCEEAcervoPortlet) -- ver comentario completo junto de
        # listar_ccee(), na celula acima. O HTML default (sem filtro) ja
        # vem com a janela recente (~30 dias) pronta, por isso usa
        # baixar_pagina() normal, sem precisar do endpoint AJAX do botao
        # "Filtrar".
        "site_url": "https://www.ccee.org.br/web/guest/atas-da-diretoria",
        "source_id": "ccee_atas_diretoria",
        "tema": "ENERGIA",
        "source_descricao": "Linked from CCEE — Atas da Diretoria",
        "listar": None,  # preenchido logo abaixo, depois de listar_ccee() ser definida
    },
}

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

PASTA_BASE = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/files/{HOJE}"

PASTA_MANIFESTOS = "/Volumes/desafio_kinea/research/research_volume/infraestrutura/manifests"
os.makedirs(PASTA_MANIFESTOS, exist_ok=True)

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
]

IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]
HTTP_TIMEOUT = 60
MIN_CHARS_TEXTO = 200


In [0]:
def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def carregar_manifesto(caminho: str) -> set:
    if not os.path.exists(caminho):
        return set()
    try:
        with open(caminho, "r", encoding="utf-8") as f:
            return set(json.load(f))
    except Exception:
        return set()


def salvar_manifesto(caminho: str, urls: set) -> None:
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(sorted(urls), f, ensure_ascii=False, indent=2)


In [0]:
def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=url)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None


def extrair_links_pdf(html: str, url_base: str) -> list[dict]:
    import urllib.parse
    soup = BeautifulSoup(html, "lxml")
    pdfs, vistos = [], set()

    for tag_a in soup.find_all("a", href=True):
        href = tag_a["href"].strip()
        if not href.lower().endswith(".pdf"):
            continue

        url_absoluta = urllib.parse.urljoin(url_base, href)
        if url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)

        titulo = tag_a.get_text(" ", strip=True)
        if not titulo:
            nome_arquivo = urllib.parse.unquote(url_absoluta.split("/")[-1])
            titulo = re.sub(r"\.pdf$", "", nome_arquivo, flags=re.IGNORECASE).replace("-", " ").replace("_", " ")

        pdfs.append({"titulo": titulo, "url": url_absoluta})

    return pdfs



# --- CCEE — Atas da Diretoria ---
#
# Liferay (portlet br_org_ccee_liferay_atas_cad_CCEEAcervoPortlet) --
# WAF bloqueia requisicao simples (403 "acesso bloqueado"), curl_cffi
# com impersonation de TLS passa normal com os mesmos headers ja usados
# no resto do projeto, sem precisar de nada adicional.
#
# O HTML default (sem nenhum filtro aplicado pelo usuario) ja vem com a
# janela recente pronta: o JS da propria pagina seta um filtro de data
# de "ultimos 30 dias" antes do primeiro paint, entao baixar_pagina()
# simples ja devolve os PDFs certos -- nao precisa do endpoint AJAX
# (Liferay serveResource, p_p_lifecycle=2) que o botao "Filtrar" dispara.
# Esse endpoint AJAX existe e devolve um arquivo bem maior (testado:
# ~460 documentos numa janela 2020-2026), mas mistura Atas da Diretoria
# com Atas do Conselho de Administracao sem filtro de tipo no request --
# por isso nao e usado aqui; se um backfill completo for pedido no
# futuro, filtrar por nomeDocumentoList == "Ata de Reuniao da Diretoria"
# nos resultados desse endpoint.
#
# Diferente das outras fontes deste dispatcher (que usam o
# extrair_links_pdf() generico), a CCEE tem um listar_ccee() proprio pra
# pegar o titulo especifico (numero/data da reuniao, em .card-subtitle)
# e a data (.card-published) em vez do texto generico do link.

def listar_ccee(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens = []

    for card in soup.select("#resultsHTML .card"):
        tag_a = card.select_one("a.card-title[href]")
        if not tag_a:
            continue

        url_pdf = tag_a["href"].strip()

        subtitulo = card.select_one(".card-subtitle")
        titulo = subtitulo.get_text(strip=True) if subtitulo else tag_a.get_text(strip=True)

        data_publicacao = None
        tag_data = card.select_one(".card-published")
        if tag_data:
            m = re.search(r"(\d{2})/(\d{2})/(\d{4})", tag_data.get_text(strip=True))
            if m:
                dia, mes, ano = m.groups()
                data_publicacao = f"{ano}-{mes}-{dia}"

        itens.append({"titulo": titulo, "url": url_pdf, "published_at": data_publicacao})

    return itens


CONFIGS_FONTES["ccee_atas_diretoria"]["listar"] = listar_ccee

In [0]:
def baixar_pdf(url: str, tentativas: int = 3) -> Optional[bytes]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios()
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            content_type = resp.headers.get("content-type", "")
            if resp.status_code == 200 and "pdf" in content_type.lower() and resp.content:
                return resp.content
        except Exception as e:
            print(f"  [pdf httpx tent {tentativa}/{tentativas}] erro: {e}")

        # Fallback com impersonation de TLS -- necessario pra fontes atras
        # de WAF (ex.: CCEE, que bloqueia requisicao simples com 403).
        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate, timeout=HTTP_TIMEOUT)
            content_type = resp.headers.get("content-type", "")
            if resp.status_code == 200 and "pdf" in content_type.lower() and resp.content:
                return resp.content
        except Exception as e:
            print(f"  [pdf curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.0))

    return None


def extrair_texto_pdf(conteudo_pdf: bytes) -> str:
    try:
        reader = PdfReader(io.BytesIO(conteudo_pdf))
        paginas = [p.extract_text() or "" for p in reader.pages]
        return "\n\n".join(paginas).strip()
    except Exception as e:
        print(f"    -> falha ao extrair texto: {e}")
        return ""


In [0]:
def salvar_artefatos(pasta: str, source_id: str, titulo: str, texto: str, metadados: dict) -> tuple[str, str]:
    slug_source = slugify(source_id, max_len=40)
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_json


def processar_pdf(item: dict, source_id: str, source_descricao: str, pasta_destino: str) -> Optional[dict]:
    titulo, url = item["titulo"], item["url"]
    print(f"\n  [pdf] {titulo[:100]}")

    conteudo = baixar_pdf(url)
    if not conteudo:
        print("    -> download falhou; pulando.")
        return None

    texto = extrair_texto_pdf(conteudo)
    if not texto or len(texto) < MIN_CHARS_TEXTO:
        print(f"    -> texto insuficiente ({len(texto)} chars); pulando.")
        return None

    metadados = {
    "source_id": source_id,
    "title": titulo,
    "description": source_descricao,
    "url": url,
    "date": HOJE,
    "published_at": HOJE,
}

    caminho_txt, caminho_json = salvar_artefatos(pasta_destino, source_id, titulo, texto, metadados)
    print(f"    -> salvo em {caminho_txt}")
    return {"titulo": titulo, "url": url, "caminho_txt": caminho_txt, "caminho_json": caminho_json}


In [0]:
if NOME_FONTE == "todas":
    fontes_a_rodar = CONFIGS_FONTES
else:
    if NOME_FONTE not in CONFIGS_FONTES:
        raise ValueError(f"Fonte {NOME_FONTE!r} não configurada. Opções: {list(CONFIGS_FONTES)}")
    fontes_a_rodar = {NOME_FONTE: CONFIGS_FONTES[NOME_FONTE]}

resumo_geral = {}

for nome_fonte, config in fontes_a_rodar.items():
    print(f"\n{'='*70}\n=== Fonte: {nome_fonte!r} ===\n{'='*70}")

    caminho_manifesto = os.path.join(PASTA_MANIFESTOS, f"{config['source_id']}_processados.json")
    ja_processados = carregar_manifesto(caminho_manifesto)

    pasta_destino_fonte = PASTA_BASE
    os.makedirs(pasta_destino_fonte, exist_ok=True)

    try:
        html = baixar_pagina(config["site_url"])
        if not html:
            resumo_geral[nome_fonte] = "ERRO: download da listagem falhou"
            atualizar_status_fonte(
                source_id=config["source_id"],
                sucesso=False,
                docs_capturados=0,
                erro="download da listagem falhou",
            )
            continue

        listar_fn = config.get("listar") or extrair_links_pdf
        pdfs_na_pagina = listar_fn(html, config["site_url"])
        pdfs_novos = [p for p in pdfs_na_pagina if p["url"] not in ja_processados]
        print(f"{len(pdfs_na_pagina)} PDFs na página, {len(pdfs_novos)} novos.")

        salvos = 0
        for item in pdfs_novos:
            resultado = processar_pdf(item, config["source_id"], config["source_descricao"], pasta_destino_fonte)
            if resultado:
                salvos += 1
                ja_processados.add(item["url"])

        salvar_manifesto(caminho_manifesto, ja_processados)
        resumo_geral[nome_fonte] = f"{salvos} novos salvos"

        atualizar_status_fonte(
            source_id=config["source_id"],
            sucesso=True,
            docs_capturados=salvos,
        )

    except Exception as e:
        resumo_geral[nome_fonte] = f"ERRO: {e}"

        atualizar_status_fonte(
            source_id=config["source_id"],
            sucesso=False,
            docs_capturados=0,
            erro=str(e),
        )

print(f"\n\n{'='*70}\n=== RESUMO ===")
for nome_fonte, resultado in resumo_geral.items():
    print(f"  {nome_fonte}: {resultado}")
